#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Step 2: Load the data
#### first we read the file and look at the shape and the first rows.

In [2]:
a = pd.read_csv("Fraud.csv")
print(a.shape)
a.head()

(30625, 11)


,Step,Type,AMOUNT,Name Orig,oldbalanceOrg,new balance Orig,NameDest,old_balance_dest,newbalanceDest,IsFraud,Is Flagged Fraud
0,522,TRANSFER,3635328.11,C743822679,10436.0,0.0,C1534408959,273252.13,3908580.24,0,0
1,42,CASH_OUT,5032484.73,C253446049,5032484.73,0.0,C57938337,873589.17,5906073.9,1,0
2,396,CASH_OUT,334672.99,C808555938,20408.0,0.0,c105578345,205414.96,540087.95,0,False
3,182,PAYMENT,18838.31,C262764700,10492.0,0.0,M2082487144,0.0,0.0,0,0
4,402,PAYMENT,4991.69,C1021808615,0.0,0.0,M1127549690,0.0,0.0,False,0


## Step 3: Remove duplicate rows
##### the same transaction repeated many times makes the model learn the same row again and again, and it can leak into the test set. We remove duplicates first, before filling any value.

In [3]:
print("Duplicate rows:", a.duplicated().sum())

Duplicate rows: 624


In [4]:
a.drop_duplicates(inplace=True)

In [5]:
print("Shape after removing duplicates:", a.shape)

Shape after removing duplicates: (30001, 11)


## Step 4: Clean the data

### # 4.1 clean column names

In [6]:
a.columns = a.columns.str.strip()

In [7]:
a = a.rename(columns={"AMOUNT": "Amount",
                      "new balance Orig": "NewbalanceOrig",
                      "Is Flagged Fraud": "IsFlaggedFraud",
                      "oldbalanceOrg":"Oldbalanceorg",
                      "old_balance_dest":"Oldbalancedest",
                      "newbalanceDest":"Newbalancedest"})

In [8]:
a.head()

,Step,Type,Amount,Name Orig,Oldbalanceorg,NewbalanceOrig,NameDest,Oldbalancedest,Newbalancedest,IsFraud,IsFlaggedFraud
0,522,TRANSFER,3635328.11,C743822679,10436.0,0.0,C1534408959,273252.13,3908580.24,0,0
1,42,CASH_OUT,5032484.73,C253446049,5032484.73,0.0,C57938337,873589.17,5906073.9,1,0
2,396,CASH_OUT,334672.99,C808555938,20408.0,0.0,c105578345,205414.96,540087.95,0,False
3,182,PAYMENT,18838.31,C262764700,10492.0,0.0,M2082487144,0.0,0.0,0,0
4,402,PAYMENT,4991.69,C1021808615,0.0,0.0,M1127549690,0.0,0.0,False,0


In [9]:
a["Step"].unique()

<ArrowStringArray>
[     '522',       '42',      '396',      '182',      '402',       '20',
  'Hour 36',      '354',      '158',      '403',
 ...
 'Hour 670', 'Hour 507',    '189.0', 'Hour 311', 'Hour 514',    '475.0',
 'Hour 277',    '712.0',    '687.0', 'Hour 276']
Length: 1398, dtype: str

##### 4.2 Step: remove the word 'Hour', convert to number, fill missing with median

In [10]:
a["Step"] = a["Step"].str.replace("Hour","",regex=False).str.strip()

In [11]:
a["Step"] = pd.to_numeric(a["Step"],errors="coerce")

In [12]:
a["Step"] = a["Step"].fillna(a["Step"].median())

In [13]:
a["Step"] = a["Step"].astype(int)

In [14]:
a["Type"].unique()

<ArrowStringArray>
[   'TRANSFER',    'CASH_OUT',  ' PAYMENT  ',     'PAYMENT', ' CASH_OUT  ',
    'cash_out',    'Cash_Out',     'CASH_IN',       'DEBIT',    'CASH OUT',
           nan,     'payment',    'transfer',     'Cash_In',    'cash-out',
     'CASH IN',     'Payment',     'cash_in',           '?',  ' CASH_IN  ',
       'debit',    'Transfer', ' TRANSFER  ',           '-',     'cash-in',
    ' DEBIT  ',       'Debit']
Length: 27, dtype: str

##### # 4.3 Type: strip spaces, upper case, same spelling, '?' and '-' become missing, fill with mode

In [15]:
a["Type"] = a["Type"].astype(str).str.strip()

In [16]:
a["Type"] = a["Type"].str.upper()

In [17]:
a["Type"] = a["Type"].str.replace(r"[\s\-]+","_",regex=True)

In [18]:
a["Type"].unique()

<ArrowStringArray>
['TRANSFER', 'CASH_OUT', 'PAYMENT', 'CASH_IN', 'DEBIT', nan, '?', '_']
Length: 8, dtype: str

In [19]:
a["Type"] = a["Type"].replace(["?", "_", ""], np.nan)

In [20]:
a["Type"] = a["Type"].fillna(a["Type"].mode()[0])

In [21]:
a["Amount"].value_counts()

Amount
?                 132
-                 120
10000000.0        107
$10,000,000.00      8
0.0                 6
                 ... 
98742.1             1
3708.48             1
234229.47           1
116530.2            1
568292.72           1
Name: count, Length: 28575, dtype: int64

#### clean and Convert on Amount Column 

In [22]:
a["Amount"] = a["Amount"].astype(str).str.strip()

In [23]:
a["Amount"] = a["Amount"].str.replace("$","",regex=False)

In [24]:
a["Amount"] = a["Amount"].str.replace(",","",regex=False)

In [25]:
a["Amount"] = pd.to_numeric(a["Amount"],errors="coerce")

In [26]:
a["Amount"].isnull().sum()

np.int64(911)

In [27]:
a["Amount"] = a["Amount"].fillna(a["Amount"].median())

In [28]:
a["Name Orig"].value_counts()

Name Orig
-              54
?              41
C743822679      1
C253446049      1
C808555938      1
               ..
C1106227555     1
C712482180      1
C57069023       1
C427871642      1
C695109550      1
Name: count, Length: 29728, dtype: int64

In [29]:
a["Name Orig"] = a["Name Orig"].astype(str).str.strip()

In [30]:
a["Name Orig"] = a["Name Orig"].str.upper()

In [31]:
a["Name Orig"] = a["Name Orig"].replace(["?", "-"], np.nan)

In [32]:
a["Name Orig"].isnull().sum()

np.int64(275)

In [33]:
a["Name Orig"] = a["Name Orig"].fillna(a["Name Orig"].mode()[0])

In [34]:
a["Name Orig"].unique()

<ArrowStringArray>
[ 'C743822679',  'C253446049',  'C808555938',  'C262764700', 'C1021808615',
 'C1748546792',  'C608421499',  'C617740104', 'C2098091153',  'C784241936',
 ...
  'C810700448', 'C2023350964',  'C779464023', 'C1881524965',  'C736287944',
 'C1106227555',  'C712482180',   'C57069023',  'C427871642',  'C695109550']
Length: 29726, dtype: str

In [35]:
a["Oldbalanceorg"].unique()

<ArrowStringArray>
[   '10436.0', '5032484.73',    '20408.0',    '10492.0',        '0.0',
     '5783.0',     '5693.0',  '176192.66',   '78424.92',  '533074.88',
 ...
    '22431.0',  '296065.36',     '2334.0',   '496952.0',  '310559.26',
   '333197.0', '5930115.05', '4216087.23',   '116530.2',  '568292.72']
Length: 18471, dtype: str

In [36]:
a["Oldbalanceorg"].value_counts()

Oldbalanceorg
0.0           8889
-               94
?               86
10000000.0      56
149.0            9
              ... 
333197.0         1
5930115.05       1
4216087.23       1
116530.2         1
568292.72        1
Name: count, Length: 18470, dtype: int64

In [37]:
a["Oldbalanceorg"] = pd.to_numeric(a["Oldbalanceorg"],errors="coerce")

In [38]:
a["Oldbalanceorg"].isnull().sum()

np.int64(587)

In [39]:
a["Oldbalanceorg"].value_counts()

Oldbalanceorg
0.00           8889
10000000.00      56
149.00            9
151.00            8
123.00            7
               ... 
333197.00         1
5930115.05        1
4216087.23        1
116530.20         1
568292.72         1
Name: count, Length: 18468, dtype: int64

In [40]:
a["Oldbalanceorg"] = a["Oldbalanceorg"].fillna(a["Oldbalanceorg"].median())

In [41]:
a["NewbalanceOrig"].value_counts()

NewbalanceOrig
0.0            17974
?                 88
-                 75
10399045.08        2
39329.79           2
               ...  
491190.66          1
387091.06          1
115080.32          1
6028857.15         1
4450316.7          1
Name: count, Length: 11438, dtype: int64

In [42]:
a["NewbalanceOrig"] = a["NewbalanceOrig"].astype(str).str.strip()

In [43]:
a["NewbalanceOrig"] = pd.to_numeric(a["NewbalanceOrig"],errors="coerce")

In [44]:
a["NewbalanceOrig"].isnull().sum()

np.int64(590)

In [45]:
a["NewbalanceOrig"].dtype

dtype('float64')

In [46]:
a["NewbalanceOrig"].isnull().sum()

np.int64(590)

In [47]:
a["NameDest"].unique()

<ArrowStringArray>
[ 'C1534408959',    'C57938337', ' c105578345 ',  'M2082487144',
  'M1127549690',  'M1803258310',  'C1722181765',   'M270280410',
  'M1523175097',  'M1655142813',
 ...
  'C1100809835',   'C619979126',  'C1165490053',   'C399754357',
  'C2025812353',  'M2020457247',  'C1033091693',   'C815018570',
   'C590191709',    'C21214474']
Length: 29056, dtype: str

In [48]:
a["NameDest"].value_counts()

NameDest
-              48
?              41
C671848738      4
C1507761286     3
C1679714664     3
               ..
M2020457247     1
C1033091693     1
C815018570      1
C590191709      1
C21214474       1
Name: count, Length: 29055, dtype: int64

In [49]:
a["NameDest"] = a["NameDest"].astype(str).str.strip().str.upper()

In [50]:
a["NameDest"] = a["NameDest"].replace(["?", "-"], np.nan)

In [51]:
a["NameDest"].isnull().sum()

np.int64(323)

In [52]:
a["NameDest"] = a["NameDest"].fillna(a["NameDest"].mode()[0])

In [53]:
a["NameDest"].unique()

<ArrowStringArray>
['C1534408959',   'C57938337',  'C105578345', 'M2082487144', 'M1127549690',
 'M1803258310', 'C1722181765',  'M270280410', 'M1523175097', 'M1655142813',
 ...
 'C1100809835',  'C619979126', 'C1165490053',  'C399754357', 'C2025812353',
 'M2020457247', 'C1033091693',  'C815018570',  'C590191709',   'C21214474']
Length: 29014, dtype: str

In [54]:
a["Oldbalancedest"].value_counts()

Oldbalancedest
0.0           13191
-                67
?                56
10000000.0        2
10596.0           2
              ...  
2045273.41        1
121497.97         1
936706.99         1
158291.79         1
6591685.44        1
Name: count, Length: 16336, dtype: int64

In [55]:
a["Oldbalancedest"] = a["Oldbalancedest"].replace(["", "?", "-"], np.nan)

In [56]:
a["Oldbalancedest"].isnull().sum()

np.int64(475)

In [57]:
a["Oldbalancedest"] = pd.to_numeric(
    a["Oldbalancedest"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce")

In [58]:
a["Oldbalancedest"] = a["Oldbalancedest"].fillna(a["Oldbalancedest"].median())

In [59]:
a["Newbalancedest"] = a["Newbalancedest"].replace([ "?", "-"], np.nan)

In [60]:
a["Newbalancedest"].isnull().sum()

np.int64(479)

In [61]:
a["Newbalancedest"] = pd.to_numeric(
    a["Newbalancedest"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce")

In [62]:
a["Newbalancedest"] = a["Newbalancedest"].fillna(a["Newbalancedest"].median())

In [63]:
a["IsFlaggedFraud"].isnull().sum()

np.int64(1)

In [64]:
a.info()

<class 'pandas.DataFrame'>
Index: 30001 entries, 0 to 30624
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Step            30001 non-null  int64  
 1   Type            30001 non-null  str    
 2   Amount          30001 non-null  float64
 3   Name Orig       30001 non-null  str    
 4   Oldbalanceorg   30001 non-null  float64
 5   NewbalanceOrig  29411 non-null  float64
 6   NameDest        30001 non-null  str    
 7   Oldbalancedest  30001 non-null  float64
 8   Newbalancedest  30001 non-null  float64
 9   IsFraud         30000 non-null  str    
 10  IsFlaggedFraud  30000 non-null  str    
dtypes: float64(5), int64(1), str(5)
memory usage: 3.6 MB


In [65]:
a["IsFraud"].unique()

<ArrowStringArray>
['0', '1', 'False', 'No', 'N', 'True', 'Y', 'Yes', nan]
Length: 9, dtype: str

In [66]:
flag = {"1": 1, "0": 0, "yes": 1, "no": 0, "y": 1, "n": 0, "true": 1, "false": 0}

In [67]:
a["IsFraud"] = a["IsFraud"].astype(str).str.strip().str.lower().map(flag)

In [68]:
a = a.dropna(subset=["IsFraud"])

In [69]:
a["IsFraud"] = a["IsFraud"].astype(int)

In [70]:
print(a["IsFraud"].value_counts())
print(a["IsFraud"].mean() * 100)

IsFraud
0    27000
1     3000
Name: count, dtype: int64
10.0


In [71]:
a["IsFlaggedFraud"] = a["IsFlaggedFraud"].astype(str).str.strip().str.lower().map(flag)

In [72]:
a["IsFlaggedFraud"] = a["IsFlaggedFraud"].fillna(0).astype(int)

In [73]:
print(pd.crosstab(a["IsFraud"], a["IsFlaggedFraud"]))

IsFlaggedFraud      0  1
IsFraud                 
0               27000  0
1                2995  5


In [74]:
print(a.shape)
print(a.isnull().sum())

(30000, 11)
Step                0
Type                0
Amount              0
Name Orig           0
Oldbalanceorg       0
NewbalanceOrig    589
NameDest            0
Oldbalancedest      0
Newbalancedest      0
IsFraud             0
IsFlaggedFraud      0
dtype: int64


In [75]:
rule = np.where(a["Type"] == "CASH_IN",
                a["Oldbalanceorg"] + a["Amount"],
                (a["Oldbalanceorg"] - a["Amount"]).clip(lower=0))
a["NewbalanceOrig"] = a["NewbalanceOrig"].fillna(pd.Series(rule, index=a.index))

In [76]:
a.isnull().sum()

Step              0
Type              0
Amount            0
Name Orig         0
Oldbalanceorg     0
NewbalanceOrig    0
NameDest          0
Oldbalancedest    0
Newbalancedest    0
IsFraud           0
IsFlaggedFraud    0
dtype: int64

In [77]:
a.info()

<class 'pandas.DataFrame'>
Index: 30000 entries, 0 to 30624
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Step            30000 non-null  int64  
 1   Type            30000 non-null  str    
 2   Amount          30000 non-null  float64
 3   Name Orig       30000 non-null  str    
 4   Oldbalanceorg   30000 non-null  float64
 5   NewbalanceOrig  30000 non-null  float64
 6   NameDest        30000 non-null  str    
 7   Oldbalancedest  30000 non-null  float64
 8   Newbalancedest  30000 non-null  float64
 9   IsFraud         30000 non-null  int64  
 10  IsFlaggedFraud  30000 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 3.6 MB


In [78]:
a.to_csv("Fraud_cleaned.csv",index=False)